# 13.2 Iterating over a set: the for statement

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The examples above still require that some command be typed repeatedly. AMPL provides
looping commands that can do this work automatically, with various options to
determine how long the looping should continue.

We begin with the for statement, which executes a statement or collection of statements
once for each member of some set. To execute our multi-period production problem
sensitivity analysis script four times, for example, we can use a single for statement
followed by the command that we want to repeat:

```ampl
ampl: model steelT.mod;
ampl: data steelT.dat;
ampl: for {1..4} commands steelT.sa1;
MINOS 5.5: optimal solution found.

15 iterations, objective 515033
MINOS 5.5: optimal solution found.

1 iterations, objective 532033
MINOS 5.5: optimal solution found.

1 iterations, objective 549033
MINOS 5.5: optimal solution found.

2 iterations, objective 565193
ampl:
```

The expression between for and the command can be any AMPL indexing expression.

As an alternative to taking the commands from a separate file, we can write them as
the body of a for statement, enclosed in braces:

```ampl
model steelT.mod;
data steelT.dat;
for {1..4} {
       solve;
       display Total_Profit >steelT.sens;
       option display_1col 0;
       option omit_zero_rows 0;
       display Make >steelT.sens;
       display Sell >steelT.sens;
       option display_1col 20;
       option omit_zero_rows 1;
       display Inv >steelT.sens;
       let avail[3] := avail[3] + 5;
}
```

If this script is stored in steelT.sa2, then the whole iterated sensitivity analysis is carried
out by typing

```ampl
ampl: commands steelT.sa2
```

This approach tends to be clearer and easier to work with, particularly as we make the
loop more sophisticated. As a first example, consider how we would go about compiling
a `table` of the `objective` and the dual value on constraint Time[3], for successive values
of avail[3]. A script for this purpose is shown in [Figure 13-1](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-1). After the `model` and
`data` are read, the script provides additional declarations for the `table` of values:

```ampl
set AVAIL3;
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
```

The set AVAIL3 will contain all the different values for avail[3] that we want to try;
for each such value a, avail3_obj[a] and avail3_dual[a] will be the associated
`objective` and dual values. Once these are set up, we assign the set value to
AVAIL3:

```ampl
let AVAIL3 := avail[3] .. avail[3] + 15 by 5;
```

and then use a for loop to iterate over this set:

```ampl
for {a in AVAIL3} {
       let avail[3] := a;
       solve;
       let avail3_obj[a] := Total_Profit;
       let avail3_dual[a] := Time[3].dual;
}
```

We see here that a for loop can be over an arbitrary set, and that the index running over
the set (a in this case) can be used in statements within the loop. After the loop is commodel

```ampl
steelT.mod;
data steelT.dat;
option solver_msg 0;
set AVAIL3;
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
let AVAIL3 := avail[3] .. avail[3] + 15 by 5;
for {a in AVAIL3} {
       let avail[3] := a;
       solve;
       let avail3_obj[a] := Total_Profit;
       let avail3_dual[a] := Time[3].dual;
}
display avail3_obj, avail3_dual;
```

**[Figure 13-1](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-1):** Parameter sensitivity script (steelT.sa3).

plete, the desired `table` is produced by displaying avail3_obj and avail3_dual, as
shown at the end of the script in [Figure 13-1](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-1). If this script is stored in steelT.sa3,
then the desired results are produced with a single command:

```ampl
ampl: commands steelT.sa3;
: avail3_obj avail3_dual :=
32    515033      3400
37    532033      3400
42    549033      3400
47    565193      2980
;
```

In this example we have suppressed the messages from the solver, by including the command
option solver_msg 0 in the script.

AMPL's for loops are also convenient for generating formatted tables. Suppose that
after solving the multi-period production problem, we want to `display` sales both in tons
and as a percentage of the market limit. We could use a `display` command to produce
a `table` like this:

```ampl
ampl: display {t in 1..T, p in PROD}
ampl?    (Sell[p,t], 100*Sell[p,t]/market[p,t]);
:       Sell[p,t] 100*Sell[p,t]/market[p,t]    :=
1 bands    6000            100
1 coils     307              7.675
2 bands    6000            100
2 coils    2500            100
3 bands    1400             35
3 coils    3500            100
4 bands    2000             30.7692
4 coils    4200            100
;
```

By writing a script that uses the printf command (A.16), we can create a much more
effective `table`:

```ampl
ampl: commands steelT.tab1;
SALES         bands            coils
week 1     6000 100.0%       307     7.7%
week 2     6000 100.0%      2500 100.0%
week 3     1399   35.0%     3500 100.0%
week 4     1999   30.8%     4200 100.0%
```

The script to write this `table` can be as short as two printf commands:

```ampl
printf "\n%s%14s%17s\n", "SALES", "bands", "coils";
printf {t in 1..T}: "week %d%9d%7.1f%%%9d%7.1f%%\n", t,
       Sell["bands",t], 100*Sell["bands",t]/market["bands",t],
       Sell["coils",t], 100*Sell["coils",t]/market["coils",t];
```

This approach is undesirably restrictive, however, because it assumes that there will
always be two products and that they will always be named coils and bands. In fact
the printf statement cannot write a `table` in which both the number of rows and the
number of columns depend on the `data`, because the number of entries in its format string
is always fixed.

A more general script for generating the `table` is shown in [Figure 13-2](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-2). Each pass
through the "outer" loop over {1..T} generates one row of the `table`. Within each
pass, an "inner" loop over PROD generates the row's product entries. There are more
printf statements than in the previous example, but they are shorter and simpler. We
use several statements to write the contents of each line; printf does not begin a new
line except where a newline (\n) appears in its format string.

Loops can be nested to any depth, and may be iterated over any set that can be represented
by an AMPL set expression. There is one pass through the loop for every member
of the set, and if the set is ordered — any set of numbers like 1..T, or a set declared
ordered or circular — the order of the passes is determined by the ordering of the
set. If the set is unordered (like PROD) then AMPL chooses the order of the passes, but

```ampl
printf "\nSALES";
printf {p in PROD}: "%14s   ", p;
printf "\n";
for {t in 1..T} {
       printf "week %d", t;
       for {p in PROD} {
       printf "%9d", Sell[p,t];
       printf "%7.1f%%", 100 * Sell[p,t]/market[p,t];
       }
       printf "\n";
}
```

**[Figure 13-2](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-2):** Generating a formatted sales `table` with nested loops (steelT.tab1).

the choice is the same every time; the [Figure 13-2](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-2) script relies on this consistency to
ensure that all of the entries in one column of the `table` refer to the same product.